In [1]:
!pip install gradio transformers gtts playsound speechrecognition

In [2]:
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from gtts import gTTS
import os
import speech_recognition as sr
import torch

In [3]:
!pip install transformers accelerate bitsandbytes
!pip install -U bitsandbytes transformers accelerate
!pip install -U bitsandbytes # Ensure bitsandbytes is up-to-date for 4-bit quantization

In [4]:
# Load model and tokenizer
model_name = "meta-llama/Llama-2-7b-chat-hf"
token = ""

# Check if CUDA is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer with token, using 4-bit quantization for memory efficiency
# Note: This requires the 'bitsandbytes' and 'accelerate' libraries to be installed.
model = AutoModelForCausalLM.from_pretrained(model_name, use_auth_token=token, load_in_4bit=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [5]:
# Function to generate chatbot response without chat history
def chatbot_response(user_input):
    # Tokenize input and move to GPU
    inputs = tokenizer(user_input, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate response
    outputs = model.generate(**inputs)

    # Decode output
    bot_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Ensure the bot doesn't repeat the user input
    if bot_reply.startswith(user_input):
        bot_reply = bot_reply[len(user_input):].strip()

    return bot_reply

In [6]:
def text_to_audio(response, voice="default"):
    # Convert text response to audio
    tts = gTTS(text=response, lang="en", slow=False)
    audio_path = "response.mp3"
    tts.save(audio_path)
    return audio_path

In [7]:
def process_input(text_input, voice_input):
    # Use speech recognition if voice input is provided
    if voice_input:
        recognizer = sr.Recognizer()
        with sr.AudioFile(voice_input) as source:
            audio = recognizer.record(source)
        try:
            # Convert speech to text
            text_input = recognizer.recognize_google(audio)
        except sr.UnknownValueError:
            text_input = "Sorry, I did not understand that."
        except sr.RequestError:
            text_input = "Sorry, there was an error with the speech service."

    # Generate AI response using chatbot_response function
    response = chatbot_response(text_input)
    audio_file = text_to_audio(response, voice_input)
    return response, audio_file

In [8]:
# Gradio interface
with gr.Blocks() as voice_assistant:
    gr.Markdown("""
    # Voice Assistant

    You can type or speak your questions, and I will respond in both text and audio.
    """)

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(label="Enter your message here:")
            voice_input = gr.Audio(type="filepath", label="Or record your voice:")
            submit_btn = gr.Button("Submit")

        with gr.Column():
            response_output = gr.Textbox(label="AI Response:")
            audio_output = gr.Audio(label="Listen to Response:")

    submit_btn.click(
        fn=process_input,
        inputs=[text_input, voice_input],
        outputs=[response_output, audio_output]
    )

In [9]:
# Launch the Gradio app
voice_assistant.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30d1a4c1fc549bcbe7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
